# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path


def percentile_rank(series: pd.Series) -> pd.Series:
    return series.rank(pct=True, method="average").fillna(0.0)


def normalize(series: pd.Series) -> pd.Series:
    series = series.astype(float)
    lo, hi = series.min(), series.max()
    if pd.isna(lo) or pd.isna(hi) or hi == lo:
        return pd.Series(0.0, index=series.index)
    return (series - lo) / (hi - lo)


def reason_codes(row: pd.Series) -> list[str]:
    reasons = []
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        reasons.append("stale_visible_page")
    if row["word_count"] > 0 and row["word_count"] < 1200 and row["impressions_90d"] >= 250:
        reasons.append("thin_visible_page")
    if row["avg_position"] > 0 and row["avg_position"] <= 10 and row["content_age_days"] >= 180:
        reasons.append("page_one_decay_risk")
    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5:
        reasons.append("low_ctr_visible_page")
    if row["sessions_90d"] >= 30 and (
        (row["engagement_rate"] > 0 and row["engagement_rate"] < 30)
        or (row["scroll_rate"] > 0 and row["scroll_rate"] < 30)
    ):
        reasons.append("low_engagement_visible_page")
    if row["trend_direction"].lower() == "down" and row["impressions_90d"] >= 100:
        reasons.append("declining_with_demand")
    if not reasons:
        reasons.append("general_refresh_review")
    return reasons


def suggested_action(reasons: list[str]) -> str:
    reasons = set(reasons)
    if "thin_visible_page" in reasons:
        return "expand_and_refresh"
    if "low_ctr_visible_page" in reasons:
        return "refresh_and_review_ctr"
    if "stale_visible_page" in reasons or "declining_with_demand" in reasons:
        return "refresh"
    return "monitor"


print("Rule (plain words): prioritize pages that are visible, stale, and showing search opportunity.")
print("Score combines visibility demand, freshness risk, position opportunity, and depth gap.")
print("Reason codes explain the exact trigger behind each recommended action.")



Rule (plain words): prioritize pages that are visible, stale, and showing search opportunity.
Score combines visibility demand, freshness risk, position opportunity, and depth gap.
Reason codes explain the exact trigger behind each recommended action.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
input_path = Path('../../data/raw/content_refresh_anonymized.csv').resolve()
output_path = Path('../outputs/baseline_action_score.csv').resolve()

raw = pd.read_csv(input_path)
ranked_queue = (
    raw
    .drop_duplicates(subset=['content_id'])
    .loc[(raw['impressions_90d'] > 0) & (raw['content_age_days'] >= 90)]
    .copy()
)

ranked_queue['visibility_score'] = percentile_rank(np.log1p(ranked_queue['impressions_90d']))
ranked_queue['freshness_risk_score'] = percentile_rank(ranked_queue['days_since_last_update'])
ranked_queue['position_opportunity_score'] = (
    (1 - normalize(ranked_queue['avg_position'].clip(lower=1, upper=50)))
    * ranked_queue['visibility_score']
    * (ranked_queue['avg_position'] > 0).astype(int)
)
ranked_queue['depth_gap_score'] = (1 - percentile_rank(ranked_queue['word_count'])) * ranked_queue['visibility_score']

ranked_queue['baseline_action_score'] = (
    0.40 * ranked_queue['visibility_score']
    + 0.30 * ranked_queue['freshness_risk_score']
    + 0.25 * ranked_queue['position_opportunity_score']
    + 0.05 * ranked_queue['depth_gap_score']
).clip(0, 1)

ranked_queue['reason_codes'] = ranked_queue.apply(reason_codes, axis=1)
ranked_queue['primary_reason_code'] = ranked_queue['reason_codes'].str[0]
ranked_queue['reason_codes'] = ranked_queue['reason_codes'].apply(lambda xs: '|'.join(xs))
ranked_queue['action'] = ranked_queue['reason_codes'].str.split('|').apply(suggested_action)
ranked_queue['baseline_rank'] = ranked_queue['baseline_action_score'].rank(method='first', ascending=False).astype(int)

ranked_queue = ranked_queue.sort_values('baseline_rank')

output_columns = [
    'content_id', 'client_id', 'baseline_rank', 'baseline_action_score',
    'action', 'primary_reason_code', 'reason_codes',
    'impressions_90d', 'clicks_90d', 'sessions_90d',
    'avg_position', 'ctr', 'engagement_rate', 'scroll_rate',
    'content_age_days', 'days_since_last_update', 'word_count',
    'trend_direction', 'trend_pct'
]

output_path.parent.mkdir(parents=True, exist_ok=True)
ranked_queue[output_columns].to_csv(output_path, index=False)

print(f'Rows scored: {len(ranked_queue):,}')
print(f'Wrote ranked queue: {output_path}')
ranked_queue[output_columns].head(5)



Rows scored: 30,000
Wrote ranked queue: /home/runner/work/flyrank-ml-internship/flyrank-ml-internship/work/outputs/baseline_action_score.csv


,content_id,client_id,baseline_rank,baseline_action_score,action,primary_reason_code,reason_codes,impressions_90d,clicks_90d,sessions_90d,avg_position,ctr,engagement_rate,scroll_rate,content_age_days,days_since_last_update,word_count,trend_direction,trend_pct
21565,content_9532f197bbc8,client_4e07408562,1,0.947603,refresh,page_one_decay_risk,page_one_decay_risk|low_engagement_visible_pag...,309192,2689,1098,2.0,0.87,8.01,28.75,445,104,NaN,down,-37.3
4644,content_4d1fe5b32dc2,client_19581e27de,2,0.941268,monitor,page_one_decay_risk,page_one_decay_risk|low_engagement_visible_page,97999,512,549,2.5,0.52,7.47,13.15,329,104,NaN,stable,-8.0
18954,content_07f2e7a6f38a,client_19581e27de,3,0.940461,monitor,page_one_decay_risk,page_one_decay_risk|low_engagement_visible_page,101078,856,780,2.7,0.85,2.05,4.60,313,104,NaN,stable,-15.8
17400,content_e5ae436f9a16,client_4e07408562,4,0.939997,refresh_and_review_ctr,page_one_decay_risk,page_one_decay_risk|low_ctr_visible_page|low_e...,117741,533,522,3.0,0.45,7.09,12.60,421,104,NaN,stable,-0.5
9348,content_3430a8b94511,client_19581e27de,5,0.939963,refresh_and_review_ctr,page_one_decay_risk,page_one_decay_risk|low_ctr_visible_page|low_e...,152617,440,534,3.3,0.29,6.18,11.04,329,104,NaN,stable,-3.5


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
top20_review = ranked_queue.head(20).copy()


def confidence_note(row: pd.Series) -> str:
    reason_count = len(str(row['reason_codes']).split('|'))
    score = row['baseline_action_score']
    if score >= 0.75 and reason_count >= 2:
        return 'High confidence: multiple strong signals and high score.'
    if score >= 0.60:
        return 'Medium confidence: score is solid but still needs human review.'
    return 'Lower confidence: treat as directional and verify manually.'


def wrong_if(row: pd.Series) -> str:
    reasons = set(str(row['reason_codes']).split('|'))
    if 'low_ctr_visible_page' in reasons:
        return 'Wrong if CTR is temporarily low from title testing or SERP layout shifts.'
    if 'thin_visible_page' in reasons:
        return 'Wrong if short content is intentional and already satisfies search intent.'
    if 'stale_visible_page' in reasons:
        return 'Wrong if the page is evergreen and still accurate without updates.'
    if 'declining_with_demand' in reasons:
        return 'Wrong if decline is seasonal and expected to recover naturally.'
    return 'Wrong if missing context (brand campaigns, migrations, or tracking artifacts) explains the signal.'


top20_review['confidence_note'] = top20_review.apply(confidence_note, axis=1)
top20_review['what_would_make_it_wrong'] = top20_review.apply(wrong_if, axis=1)

review_columns = [
    'baseline_rank', 'content_id', 'action', 'primary_reason_code',
    'confidence_note', 'what_would_make_it_wrong', 'baseline_action_score'
]

top20_review = top20_review[review_columns]
review_path = Path('../outputs/baseline_top20_review.csv').resolve()
top20_review.to_csv(review_path, index=False)

print(f'Wrote top-20 review: {review_path}')
top20_review



Wrote top-20 review: /home/runner/work/flyrank-ml-internship/flyrank-ml-internship/work/outputs/baseline_top20_review.csv


,baseline_rank,content_id,action,primary_reason_code,confidence_note,what_would_make_it_wrong,baseline_action_score
21565,1,content_9532f197bbc8,refresh,page_one_decay_risk,High confidence: multiple strong signals and h...,Wrong if decline is seasonal and expected to r...,0.947603
4644,2,content_4d1fe5b32dc2,monitor,page_one_decay_risk,High confidence: multiple strong signals and h...,"Wrong if missing context (brand campaigns, mig...",0.941268
18954,3,content_07f2e7a6f38a,monitor,page_one_decay_risk,High confidence: multiple strong signals and h...,"Wrong if missing context (brand campaigns, mig...",0.940461
17400,4,content_e5ae436f9a16,refresh_and_review_ctr,page_one_decay_risk,High confidence: multiple strong signals and h...,Wrong if CTR is temporarily low from title tes...,0.939997
9348,5,content_3430a8b94511,refresh_and_review_ctr,page_one_decay_risk,High confidence: multiple strong signals and h...,Wrong if CTR is temporarily low from title tes...,0.939963
25409,6,content_cbd93118300b,refresh_and_review_ctr,page_one_decay_risk,High confidence: multiple strong signals and h...,Wrong if CTR is temporarily low from title tes...,0.939665
18458,7,content_9c195417f6ef,monitor,page_one_decay_risk,High confidence: multiple strong signals and h...,"Wrong if missing context (brand campaigns, mig...",0.939353
13306,8,content_ba2acb4ebd04,monitor,page_one_decay_risk,High confidence: multiple strong signals and h...,"Wrong if missing context (brand campaigns, mig...",0.938024
28354,9,content_79b25654070a,refresh_and_review_ctr,page_one_decay_risk,High confidence: multiple strong signals and h...,Wrong if CTR is temporarily low from title tes...,0.937766
8275,10,content_adddad39251c,monitor,page_one_decay_risk,High confidence: multiple strong signals and h...,"Wrong if missing context (brand campaigns, mig...",0.937520


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
weak_picks = top20_review[
    top20_review['confidence_note'].str.contains('Lower confidence', case=False)
    | top20_review['primary_reason_code'].eq('general_refresh_review')
].copy()

print('Weak picks in top-20 (need extra manual validation):')
if weak_picks.empty:
    print('- None in the current top-20 slice.')
else:
    display(weak_picks[['baseline_rank', 'content_id', 'action', 'primary_reason_code', 'confidence_note']])

used_score_columns = {
    'impressions_90d', 'days_since_last_update', 'avg_position', 'word_count',
    'ctr', 'sessions_90d', 'engagement_rate', 'scroll_rate', 'content_age_days', 'trend_direction'
}

product_flag_columns = {'health_score', 'priority_score', 'action_type', 'refresh_tier'}
future_window_columns = {'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d'}

print('\nLeakage checks:')
print(f"- Product flags used in rule: {sorted(used_score_columns & product_flag_columns)}")
print(f"- Future-window columns used in rule: {sorted(used_score_columns & future_window_columns)}")
print('- Verdict: no product flags or future-window metrics were used in the baseline rule.')
print('- Note: trend_direction is used as a current-state reason code only, not as a weighted score component.')



Weak picks in top-20 (need extra manual validation):
- None in the current top-20 slice.

Leakage checks:
- Product flags used in rule: []
- Future-window columns used in rule: []
- Verdict: no product flags or future-window metrics were used in the baseline rule.
- Note: trend_direction is used as a current-state reason code only, not as a weighted score component.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.